# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata

hf_token = userdata.get("FlyRank")
import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content,
    COUNT(DISTINCT report_date) AS unique_dates
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────┬──────────────┐
│ total_rows │ unique_content │ unique_dates │
│   int64    │     int64      │    int64     │
├────────────┼────────────────┼──────────────┤
│   78835655 │         427292 │          520 │
└────────────┴────────────────┴──────────────┘

In [4]:
# Verify that every content_hash_id exists across the full reporting period
con.sql(f"""
SELECT
    content_hash_id,
    COUNT(*) AS rows_per_content
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
GROUP BY content_hash_id
ORDER BY rows_per_content DESC
LIMIT 10;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬──────────────────┐
│     content_hash_id      │ rows_per_content │
│         varchar          │      int64       │
├──────────────────────────┼──────────────────┤
│ content_e79531cf26443659 │              520 │
│ content_a8b8771f27de20f6 │              520 │
│ content_0672d8db776419c0 │              520 │
│ content_17c76438024dece9 │              520 │
│ content_f3a75d8cf58dd50b │              520 │
│ content_0642dc7f62d4f780 │              520 │
│ content_5e770041ee8f2231 │              520 │
│ content_5175438fecb054a4 │              520 │
│ content_de7b08874af74c00 │              520 │
│ content_b7182f464dcd1e73 │              520 │
├──────────────────────────┴──────────────────┤
│ 10 rows                           2 columns │
└─────────────────────────────────────────────┘

In [5]:
con.sql(f"""
SELECT
    month,
    COUNT(DISTINCT report_date) AS days
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
GROUP BY month
ORDER BY month;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────┐
│  month  │ days  │
│ varchar │ int64 │
├─────────┼───────┤
│ 2025-01 │     5 │
│ 2025-02 │    28 │
│ 2025-03 │    31 │
│ 2025-04 │    30 │
│ 2025-05 │    31 │
│ 2025-06 │    30 │
│ 2025-07 │    31 │
│ 2025-08 │    31 │
│ 2025-09 │    30 │
│ 2025-10 │    31 │
│ 2025-11 │    30 │
│ 2025-12 │    31 │
│ 2026-01 │    31 │
│ 2026-02 │    28 │
│ 2026-03 │    31 │
│ 2026-04 │    30 │
│ 2026-05 │    31 │
│ 2026-06 │    30 │
├─────────┴───────┤
│     18 rows     │
└─────────────────┘

**Unit of analysis:**

One row represents one content item (content_hash_id) on one report_date.

**Time window:**

The dataset covers January 2025 through June 2026.
The first month is partial (5 days), while the following months contain their expected number of reporting days.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

The dataset columns are grouped into four categories based on their role in analysis.

- **Features:** Metrics that describe search, traffic, engagement, and AI activity.
- **Label:** No prediction target exists in this dataset.
- **Context:** Metadata that provides additional information about each observation.
- **Excluded:** Identifier columns that should not be used as model features.

In [6]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

### Feature

- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position
- ga4_pageviews
- ga4_engaged_sessions
- scroll_events
- sessions_ai
- ai_chatgpt
- ai_gemini
- ai_claude
- ai_copilot
- ai_perplexity
- ai_meta
- ai_other

These columns are measurable metrics describing search performance, user engagement, and AI-driven traffic.

---

### Label

No explicit prediction label exists in this dataset.

---

### Context

- report_date
- month
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

These columns describe the reporting context rather than the measured outcomes.

---

### Excluded

- client_hash_id
- content_hash_id

These are identifier columns used only to uniquely identify clients and content. They are excluded because they do not carry predictive information.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verify the dataset grain by comparing the total number of rows with the number of unique content items and reporting dates.

In [8]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content,
    COUNT(DISTINCT report_date) AS unique_dates
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────┬──────────────┐
│ total_rows │ unique_content │ unique_dates │
│   int64    │     int64      │    int64     │
├────────────┼────────────────┼──────────────┤
│   78835655 │         427292 │          520 │
└────────────┴────────────────┴──────────────┘

Check for duplicate rows at the expected grain (content × report_date).

In [9]:
con.sql(f"""
SELECT
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
GROUP BY content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 10;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬─────────────┬───────────┐
│     content_hash_id      │ report_date │ row_count │
│         varchar          │    date     │   int64   │
├──────────────────────────┼─────────────┼───────────┤
│ content_034d4ed04b3ddd55 │ 2026-06-14  │         2 │
│ content_103c6f04a93b2693 │ 2026-06-14  │         2 │
│ content_fc4ff20f189e0aa0 │ 2026-06-16  │         2 │
│ content_ef8338f4365f7423 │ 2026-06-20  │         2 │
│ content_b40e27e07768f907 │ 2026-06-16  │         2 │
│ content_99a3e159bf3e5306 │ 2026-06-16  │         2 │
│ content_023c4648db30b8c3 │ 2026-06-17  │         2 │
│ content_19da6521d25c923a │ 2026-06-17  │         2 │
│ content_0a21add649629840 │ 2026-06-20  │         2 │
│ content_dacee0e0bac2dd31 │ 2026-06-19  │         2 │
├──────────────────────────┴─────────────┴───────────┤
│ 10 rows                                  3 columns │
└────────────────────────────────────────────────────┘

Some duplicate (content_hash_id, report_date) combinations exist. Therefore, the expected grain is not perfectly unique and duplicate records should be considered during analysis.

Measure missing values for important identifier and metric columns.

In [12]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(content_hash_id) AS missing_content_id,
    COUNT(*) - COUNT(report_date) AS missing_report_date,
    COUNT(*) - COUNT(gsc_impressions) AS missing_gsc_impressions,
    COUNT(*) - COUNT(ga4_pageviews) AS missing_ga4_pageviews
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬─────────────────────┬─────────────────────────┬───────────────────────┐
│ total_rows │ missing_content_id │ missing_report_date │ missing_gsc_impressions │ missing_ga4_pageviews │
│   int64    │       int64        │        int64        │          int64          │         int64         │
├────────────┼────────────────────┼─────────────────────┼─────────────────────────┼───────────────────────┤
│   78835655 │                  0 │                   0 │                   98006 │              29635327 │
└────────────┴────────────────────┴─────────────────────┴─────────────────────────┴───────────────────────┘

Verify the reporting period covered by the dataset.

In [10]:
con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(DISTINCT report_date) AS reporting_days
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────────┐
│ start_date │  end_date  │ reporting_days │
│    date    │    date    │     int64      │
├────────────┼────────────┼────────────────┤
│ 2025-01-27 │ 2026-06-30 │            520 │
└────────────┴────────────┴────────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



This dataset has several limitations:

- January 2025 contains only 5 reporting days, so the first month is incomplete.
- Some metrics depend on Google Search Console (GSC) and Google Analytics 4 (GA4) availability.
- The dataset contains observational metrics only and cannot explain causal relationships.
- There is no explicit prediction label in the dataset.
- Identifier columns (client_hash_id and content_hash_id) should not be used as predictive features.

In [7]:
con.sql(f"""
SELECT
    month,
    COUNT(DISTINCT report_date) AS reporting_days
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
GROUP BY month
ORDER BY month;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────────┐
│  month  │ reporting_days │
│ varchar │     int64      │
├─────────┼────────────────┤
│ 2025-01 │              5 │
│ 2025-02 │             28 │
│ 2025-03 │             31 │
│ 2025-04 │             30 │
│ 2025-05 │             31 │
│ 2025-06 │             30 │
│ 2025-07 │             31 │
│ 2025-08 │             31 │
│ 2025-09 │             30 │
│ 2025-10 │             31 │
│ 2025-11 │             30 │
│ 2025-12 │             31 │
│ 2026-01 │             31 │
│ 2026-02 │             28 │
│ 2026-03 │             31 │
│ 2026-04 │             30 │
│ 2026-05 │             31 │
│ 2026-06 │             30 │
├─────────┴────────────────┤
│ 18 rows        2 columns │
└──────────────────────────┘

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.